# 01 — Python Engineering Refresher

**Goal:** Review Python features essential for production NLP code.

We'll cover:
1. Type hints and dataclasses
2. List/dict comprehensions and generators
3. File I/O and context managers
4. Error handling best practices
5. Functions as first-class objects
6. A mini resume parser using what we learn

## 1. Type Hints — Self-Documenting Code

In [ ]:
from typing import List, Dict, Optional, Tuple, Union

# Without type hints — what does this return?
def extract_skills(text):
    return text.split(",")

# With type hints — clear intent
def extract_skills_typed(text: str) -> List[str]:
    """Extract comma-separated skills from text."""
    return [s.strip() for s in text.split(",") if s.strip()]

# Optional and Union for nullable fields
def find_email(text: str) -> Optional[str]:
    """Return email if found, None otherwise."""
    import re
    match = re.search(r"[\w.+-]+@[\w-]+\.[\w.]+", text)
    return match.group(0) if match else None

print(extract_skills_typed("Python, NLP, Machine Learning"))
print(f"Email found: {find_email('Contact: john@example.com')}")
print(f"Email found: {find_email('No email here')}")

## 2. Dataclasses — Clean Data Containers

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Skill:
    name: str
    category: str = "technical"
    confidence: float = 0.0

@dataclass
class Resume:
    name: str
    email: Optional[str] = None
    skills: List[Skill] = field(default_factory=list)

    def add_skill(self, name: str, category: str = "technical",
                  confidence: float = 0.0) -> None:
        self.skills.append(Skill(name, category, confidence))

    def top_skills(self, threshold: float = 0.8) -> List[Skill]:
        return [s for s in self.skills if s.confidence >= threshold]

# Use it
resume = Resume("Srivatsa", "srivatsa@example.com")
resume.add_skill("Python", confidence=0.95)
resume.add_skill("NLP", confidence=0.88)
resume.add_skill("Java", confidence=0.60)
print(resume)
print(f"Top skills: {[s.name for s in resume.top_skills()]}")

## 3. List Comprehensions — Fast & Readable

In [ ]:
# Traditional loop
skills = [" Python ", "NLP ", " Machine Learning ", ""]
cleaned = []
for s in skills:
    s = s.strip()
    if s:
        cleaned.append(s)
print("Loop:", cleaned)

# List comprehension (faster, cleaner)
cleaned = [s.strip() for s in skills if s.strip()]
print("Comprehension:", cleaned)

# Dict comprehension
skill_lengths = {s: len(s) for s in cleaned}
print("Lengths:", skill_lengths)

# Set comprehension
unique_chars = {c for s in cleaned for c in s.lower()}
print(f"Unique chars across skills: {len(unique_chars)}")

## 4. Generators — Memory-Efficient Processing

In [ ]:
def read_chunks(text: str, chunk_size: int = 50):
    """Yield chunks of text without loading everything into memory."""
    for i in range(0, len(text), chunk_size):
        yield text[i:i + chunk_size]

# Generator expression (lazy)
large_text = "Python is great for NLP. " * 1000
chunks = read_chunks(large_text, 100)
print(f"Type: {type(chunks)}")
print(f"First chunk: {next(chunks)}")
print(f"Second chunk: {next(chunks)}")

# Count without building a list
word_count = sum(1 for chunk in read_chunks(large_text, 50) if "NLP" in chunk)
print(f"Chunks containing 'NLP': {word_count}")

## 5. File I/O — Reading Resumes

In [ ]:
# Context manager handles cleanup automatically
sample_text = """Name: John Doe
Email: john@example.com
Skills: Python, NLP, Machine Learning, TensorFlow
Experience: 5 years as Data Scientist
"""

# Write sample
with open("/tmp/sample_resume.txt", "w") as f:
    f.write(sample_text)

# Read it back
with open("/tmp/sample_resume.txt", "r") as f:
    content = f.read()
    lines = f.readlines()  # already consumed! (demonstrating file pointer)

# Read properly
with open("/tmp/sample_resume.txt", "r") as f:
    for line in f:  # lazy iteration
        print(repr(line.strip()))

## 6. Error Handling — Robust Extraction

In [ ]:
def safe_extract_email(text: str) -> Optional[str]:
    """Extract email with proper error handling."""
    try:
        if not isinstance(text, str):
            raise TypeError(f"Expected string, got {type(text)}")
        if not text.strip():
            raise ValueError("Empty text")
        import re
        match = re.search(r"[\w.+-]+@[\w-]+\.[\w.]+", text)
        return match.group(0) if match else None
    except (TypeError, ValueError) as e:
        print(f"Warning: {e}")
        return None
    except Exception as e:
        print(f"Unexpected error: {e}")
        return None

# Test cases
print(safe_extract_email("Email: test@example.com"))    # ✓
print(safe_extract_email(""))                            # Warning
print(safe_extract_email(123))                           # Warning

## 7. Functions as Objects — Pipeline Pattern

In [ ]:
from typing import Callable, Any

# Functions are first-class — pass them around
def clean_text(text: str) -> str:
    return text.strip().lower()

def remove_numbers(text: str) -> str:
    import re
    return re.sub(r"\d+", "", text)

def build_pipeline(*functions: Callable) -> Callable:
    """Compose multiple processing functions."""
    def pipeline(text: str) -> str:
        result = text
        for func in functions:
            result = func(result)
        return result
    return pipeline

# Build and use
cleaner = build_pipeline(clean_text, remove_numbers)
result = cleaner("  Hello 123 World!  ")
print(f"Cleaned: '{result}'")

## 8. Mini Resume Info Extractor

Putting it all together:

In [ ]:
import re
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class ContactInfo:
    name: Optional[str] = None
    email: Optional[str] = None
    phone: Optional[str] = None

@dataclass
class SimpleResume:
    raw_text: str
    contact: ContactInfo = field(default_factory=ContactInfo)
    skills: List[str] = field(default_factory=list)

def extract_name(text: str) -> Optional[str]:
    """Simple heuristic: first line often has the name."""
    first_line = text.strip().split("\n")[0].strip()
    if first_line and len(first_line.split()) in [2, 3]:
        return first_line
    return None

def extract_contact(text: str) -> ContactInfo:
    info = ContactInfo()
    info.name = extract_name(text)
    email_match = re.search(r"[\w.+-]+@[\w-]+\.[\w.]+", text)
    info.email = email_match.group(0) if email_match else None
    phone_match = re.search(r"[+]?[\d\s()-]{7,}", text)
    info.phone = phone_match.group(0).strip() if phone_match else None
    return info

def extract_skills(text: str, skill_list: List[str]) -> List[str]:
    """Find known skills in text."""
    text_lower = text.lower()
    found = []
    for skill in skill_list:
        if skill.lower() in text_lower:
            found.append(skill)
    return found

# Test it
sample = """Srivatsa Gorti
srivatsa@email.com | +91-9876543210

Professional Summary
Data scientist with 3 years experience in Python, NLP and ML.

Skills: Python, NLP, Machine Learning, SQL, TensorFlow
"""

known_skills = ["Python", "NLP", "Machine Learning", "SQL", "TensorFlow",
                "Java", "Spark", "Docker", "Kubernetes"]

resume = SimpleResume(raw_text=sample)
resume.contact = extract_contact(sample)
resume.skills = extract_skills(sample, known_skills)

print(f"Name:   {resume.contact.name}")
print(f"Email:  {resume.contact.email}")
print(f"Phone:  {resume.contact.phone}")
print(f"Skills: {resume.skills}")

## Summary

Today you learned:
- ✅ Type hints and dataclasses for clean code
- ✅ List/dict comprehensions (faster than loops)
- ✅ Generators for memory efficiency
- ✅ Context managers for safe I/O
- ✅ Error handling patterns
- ✅ Functions as first-class objects → pipeline pattern

These patterns will be used in every notebook going forward.